In [2]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
model = ChatOllama(model='qwen2.5:3b')
# model = ChatOllama(model='gemma3:1b', num_predict=5, temperature=1.0)   #num_predict instead of max_toxens for chatollama
response = model.invoke('Which animal has a hump on its back?')
response.content

'The animal with a prominent hump on its back is the camel. Camels have a very distinctive and useful hump located on their backs, which helps them store fat for energy when food might be scarce in their desert habitats.'

In [4]:
conversation = [
    {"role":"system", "content":"You are a helpful assistant that translates English to Hindi."},
    {"role":"user", "content":"Translate: You are a girl."},
    {"role":"assistant", "content":"Tum ek ladki ho."},
    {"role":"user", "content":"Translate: You are not a girl."},
]
response = model.invoke(conversation)
response.content

'तुम्हें लडकिया नहीं हो।'

In [5]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
conversation = [
    SystemMessage("You are a helpful assistant that translates English to Hindi."),
    HumanMessage("Translate: You are a girl."),
    AIMessage("Tum ek ladki ho."),
    HumanMessage("Translate: You are not a girl.")
]
response = model.invoke(conversation)
response.content

'तुम एक महिला नहीं हो।'

In [6]:
full = None
for chunk in model.stream("Which animal has a hump on its back?"):
    full = chunk if full is None else full+chunk
    # print(chunk.text, end="|", flush=True)
    print(full.content)

print(full.content_blocks)

The
The animal
The animal with
The animal with a
The animal with a h
The animal with a hump
The animal with a hump on
The animal with a hump on its
The animal with a hump on its back
The animal with a hump on its back is
The animal with a hump on its back is the
The animal with a hump on its back is the camel
The animal with a hump on its back is the camel (
The animal with a hump on its back is the camel (camel
The animal with a hump on its back is the camel (camelus
The animal with a hump on its back is the camel (camelus).
The animal with a hump on its back is the camel (camelus).
The animal with a hump on its back is the camel (camelus).
[{'type': 'text', 'text': 'The animal with a hump on its back is the camel (camelus).'}]


In [7]:
# async for event in model.astream_events('Hello'):
#     if event['event'] == "on_chat_model_start":
#         print(f'Input: {event['data']['input']}')
#     elif event['event'] == 'on_chat_model_stream':
#         print(f'Token: {event['data']['chunk'].text}')
#     elif event['event'] == 'on_chat_model_end':
#         print(f"Full message: {event['data']['output'].text}")
#     else:
#         pass

In [8]:
# responses = model.batch([
#     "Which animal has a hump on its back?",
#     "Which animal has a really long neck?",
#     "Which animal is the tallest in the world?"
# ])
# for response in responses:
#     print(response)

for response in model.batch_as_completed([
    "Which animal has a hump on its back?",
    "Which animal has a really long neck?",
    "Which animal is the tallest in the world?",
    "Why do parrots have colorful feathers?"
]):
    print(response)

(0, AIMessage(content='The animal with a hump on its back is the camel.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-13T14:21:30.3598105Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1765191700, 'load_duration': 258325600, 'prompt_eval_count': 39, 'prompt_eval_duration': 108146500, 'eval_count': 14, 'eval_duration': 1372659400, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019bb7bb-ac8d-7b61-be19-1a6d595d990c-0', usage_metadata={'input_tokens': 39, 'output_tokens': 14, 'total_tokens': 53}))
(1, AIMessage(content='The animal that is known for having a very long neck is the giraffe.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-13T14:21:32.6384353Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4039308000, 'load_duration': 254380400, 'prompt_eval_count': 37, 'prompt_eval_duration': 551342600, 'eval_count': 17, 'eval_duration': 1700375400, 'model_name': 'qwen

In [9]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])
result = model_with_tools.invoke("What's the weather like in Pune?")
print(result)
for tool_call in result.tool_calls:
    print(f'Tool: {tool_call["name"]}')
    print(f'Args: {tool_call["args"]}')

content='' additional_kwargs={} response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-13T14:22:23.9255713Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14431383400, 'load_duration': 212453000, 'prompt_eval_count': 158, 'prompt_eval_duration': 12040451900, 'eval_count': 21, 'eval_duration': 2139998500, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'} id='lc_run--019bb7bc-4c4f-78e0-8df2-832ca8108f45-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '3e3f9854-c948-4940-a065-6f618b204e6b', 'type': 'tool_call'}] usage_metadata={'input_tokens': 158, 'output_tokens': 21, 'total_tokens': 179}
Tool: get_weather
Args: {'location': 'Pune'}


In [10]:
messages = [{'role':'user', 'content':"What's the weather like in Pune?"}]
ai_message = model_with_tools.invoke(messages)
# print(ai_message)
messages.append(ai_message)
for tool_call in ai_message.tool_calls:
    if tool_call['name']=='get_weather':
        tool_result = get_weather.invoke(tool_call)
    # print(tool_result)
    messages.append(tool_result)
final_result = model_with_tools.invoke(messages)
final_result.content

'The weather in Pune is sunny.'

In [11]:
response = model_with_tools.invoke("What's the weather like in Pune and Goa?")
print(response.tool_calls)                         # multiple tool calls
results = []
for tool_call in response.tool_calls:
    if tool_call["name"]=='get_weather':
        result = get_weather.invoke(tool_call)
    results.append(result)
    print(result)

[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': 'f2abe086-cae1-4835-9bdc-30f1a279adf2', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Goa'}, 'id': 'a69da7a7-bad1-4b03-954c-05c6c37edfd6', 'type': 'tool_call'}]
content="It's sunny in Pune" name='get_weather' tool_call_id='f2abe086-cae1-4835-9bdc-30f1a279adf2'
content="It's sunny in Goa" name='get_weather' tool_call_id='a69da7a7-bad1-4b03-954c-05c6c37edfd6'


In [15]:
# for chunk in model_with_tools.stream("What's the weather in Pune and Goa?"):
#     # print(chunk)
#     for tool_chunk in chunk.tool_call_chunks:
#         if name := tool_chunk.get('name'):
#             print(f'Tool: {name}')
#         if id_ := tool_chunk.get('id'):
#             print(f'ID: {id_}')
#         if args := tool_chunk.get('args'):
#             print(f'Args: {args}')

gathered = None
for chunk in model_with_tools.stream("What's the weather in Pune and Goa?"):
    gathered = chunk if gathered is None else gathered+chunk
    print(gathered.tool_calls)

[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '8eee8381-e0a0-42c2-a8e9-90c801e1cdf9', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '8eee8381-e0a0-42c2-a8e9-90c801e1cdf9', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Goa'}, 'id': 'e8ff62f8-07e2-4c34-b7cc-08d29aea120c', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '8eee8381-e0a0-42c2-a8e9-90c801e1cdf9', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Goa'}, 'id': 'e8ff62f8-07e2-4c34-b7cc-08d29aea120c', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '8eee8381-e0a0-42c2-a8e9-90c801e1cdf9', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Goa'}, 'id': 'e8ff62f8-07e2-4c34-b7cc-08d29aea120c', 'type': 'tool_call'}]
